<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/main/FIV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# README: This is my dinosaur identifier AI model, built in PyTorch via Google Colab; pretty self explanatory,
# so I'll just cut to the chase and show you how the whole thing works:

In [2]:
# CONFIGURATIONS
imagesizing = (224, 224, 3)    # PyTorch image sizing requirement
confidencelevel = (0.85, 0.50) # Confidence thresholds: if confidencelevel, the level of confidence the model has -
                               # in its prediction is 0.85/85% or more, it confirms that exact species. -
                               # Otherwise, graceful fallback is triggered (e.g. Ouranosaurus would simply be -
                               # classified as "unidentified basal hadrosauriform" [or simpler if needbe].)

# safeguard-1: confirms tuples containing image sizing and confidence intervals.
def safeguard1():
  print(imagesizing , confidencelevel)

safeguard1()

# requesting data from the Paleobiology Database's (PBDB) live API; I'm using -
# only accepted species of dinosaur for this project.

import requests
DINO_INFO_url = "https://paleobiodb.org/data1.2/taxa/list.json?base_name=Dinosauria&status=accepted"
def getdinosauria(DINO_INFO_url):
  response = requests.get(DINO_INFO_url)
  dinodata = response.json()
  return dinodata

dinodata = getdinosauria(DINO_INFO_url)
print(dinodata["records"][69])

# parses through data

dinotax2 = {}
for records in dinodata["records"]:
  txnID = records["oid"]
  dinotax2[txnID] = {
      "name": records.get("nam"),
      "rank": records.get("rnk"),
      "parent": records.get("par")
  }

def fallingback(txnID):
  fallbackranks1 = ["family" , "species" , "genus" , "superfamily" , "subfamily" , "infraorder"]

  while True:
    records2 = dinotax2.get(txnID)
    if records2 is None:
      return None, None

    parentID = records2.get("parent")
    currentrank = records2.get("rank")
    currentname = records2.get("name")

    if currentrank in fallbackranks1:
        return currentname, currentrank
    if parentID is None:
        return currentname, None

    txnID = parentID


# safeguard
print(f"{len(dinotax2)} taxons identified")

samplekey1 = list(dinotax2.keys())[67]
print("random taxon:" , dinotax2[samplekey1])

# conditionals concerning previously defined confidence thresholds, classifying -
# specimens, and fallback
highconf = confidencelevel[0]
lowconf = confidencelevel[1]


# how the model guesses
def guessing (species, confscore, txnID):
  if confscore >= highconf:
    print(f"{confscore * 100}% confident of {species}")
    return species, confscore
  elif confscore >= lowconf:
    print(f"{confscore * 100}% confident of {species}")
    return species, confscore
  else:
    fallbackname , fallbackrank = fallingback(txnID)
    print(f"{confscore * 100}% confidence, unable to identify; unidentified {fallbackrank} ({fallbackname})")
    return fallbackname, confscore

# another safeguard
sample_key = list(dinotax2.keys())[235]
sample_species_name = dinotax2[sample_key]["name"]
guessing(sample_species_name, 0.28, sample_key)
print("THE ABOVE IS A SAMPLE GUESS!!")

# neural network, image processing

import torch
import torch.nn as neuralnetwork
import torchvision.models as TVmodels
import torchvision.datasets as dataset
datapath = ""
from torchvision import transforms
from torch.utils.data import DataLoader

#pytorch standard configs

datatransformation = transforms.Compose([
    transforms.Resize((224 , 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])
training = dataset.ImageFolder(root = datapath, transform = datatransformation)
speciesamt = len(training.classes)

# another safeguard
print(f"{speciesamt} species found for training")

# data loader
loader = DataLoader(training, batch_size = 50, shuffle  = True)

# resnet model

RESNETMODEL = TVmodels.resnet18(weights = TVmodels.ResNet18_Weights.DEFAULT)

# dynamic count
features = RESNETMODEL.fc.in_features
RESNETMODEL.fc = neuralnetwork.Linear(features , speciesamt)
# CUDA
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESNETMODEL = RESNETMODEL.to(CUDA)
# loss function and optimization
crit = neuralnetwork.CrossEntropyLoss()
optimizer = torch.optim.AdamW(RESNETMODEL.parameters(), lr = 0.001)
# training loop

epochsnum = 25
# another safeguard
print(f"training on {CUDA}")


for epoch in range(epochsnum):
  RESNETMODEL.train()
  runningloss = 0.0
  correctpredictions = 0
  totalpredictions = 0
for inputs, labels in loader:
  inputs = inputs.to(CUDA)
  labels = labels.to(CUDA)
  optimizer.zero_grad()
  outputs = RESNETMODEL(inputs)
  loss = crit(outputs, labels)
  loss.backward()
  optimizer.step()



(224, 224, 3) (0.85, 0.5)
{'oid': 'txn:92022', 'flg': 'IF', 'rnk': 'species', 'nam': 'Coelurosaurichnus largentierensis', 'par': 'txn:68186', 'rid': 'ref:67993', 'ext': '0', 'noc': 2}
11148 taxons identified
random taxon: {'name': 'Coelurosaurichnus (Dinosaurichnium) schlehenbergense', 'rank': 'species', 'parent': 'txn:141557'}
28.000000000000004% confidence, unable to identify; unidentified species (Megaloolithus petralta)
THE ABOVE IS A SAMPLE GUESS!!


FileNotFoundError: [Errno 2] No such file or directory: ''

In [1]:
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.6/435.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 24.2 MB/s eta 0:00:00
